# 🔤 Trie (Árbol de Prefijos) — Netflix EDA

**Autocompletado de Búsqueda en la Plataforma**  
Curso: Estructuras de Datos y Algoritmos | UTEC

---

## ¿Qué es un Trie?

Un **Trie** (pronunciado 'try', del inglés re**trie**val) es un árbol donde:
- Cada nodo representa un carácter
- Los caminos desde la raíz forman palabras
- Todos los hijos de un nodo comparten el mismo prefijo

### Complejidad vs Alternativas

| Operación | Lista Ordenada | Árbol Binario (BST) | **Trie** |
|-----------|---------------|---------------------|----------|
| Búsqueda exacta | O(log n) | O(log n) | **O(m)** |
| Autocompletado | O(n) | O(n) | **O(m + k)** |

donde `m` = longitud del prefijo, `n` = número de palabras, `k` = resultados.

### Uso en Netflix

Cuando escribes en el buscador de Netflix, el sistema sugiere títulos en **tiempo real** (<100ms). Esto es posible gracias al Trie que:
1. Navega al nodo del prefijo en O(m)
2. Recolecta todas las palabras del subárbol
3. Ordena por popularidad y retorna top-K

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/Guido-Silva/netflix-streaming-eda.git
    %cd netflix-streaming-eda
    PROJECT_ROOT = Path.cwd().resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INFORME_DIR = PROJECT_ROOT / "informe"
INFORME_DIR.mkdir(parents=True, exist_ok=True)

print(f"Entorno configurado | PROJECT_ROOT={PROJECT_ROOT}")


In [ ]:
from src.trie import Trie, TrieNode
print('✅ Módulo Trie importado')

In [ ]:
from src.trie import demo
demo()

In [ ]:
# Experimento: Tiempo de búsqueda vs tamaño del Trie
import random
import time
import string

print('⏱️  Experimento: Tiempo de Búsqueda vs Tamaño del Trie')
print('=' * 60)

# Generar palabras aleatorias de longitud variable
def generar_palabra(longitud_min=4, longitud_max=20):
    longitud = random.randint(longitud_min, longitud_max)
    return ''.join(random.choices(string.ascii_lowercase + ' ', k=longitud)).strip()

random.seed(42)
tamaños = [100, 1000, 10000, 50000]
tiempos_insert = []
tiempos_search = []
tiempos_autocomplete = []

for n in tamaños:
    palabras = [generar_palabra() for _ in range(n)]
    popularidades = [random.randint(1, 1000000) for _ in range(n)]
    
    trie = Trie()
    
    # Medir tiempo de inserción
    inicio = time.perf_counter()
    for palabra, pop in zip(palabras, popularidades):
        trie.insert(palabra, pop)
    t_insert = time.perf_counter() - inicio
    tiempos_insert.append(t_insert)
    
    # Medir tiempo de búsqueda (100 búsquedas)
    inicio = time.perf_counter()
    for palabra in random.sample(palabras, min(100, n)):
        trie.search(palabra)
    t_search = (time.perf_counter() - inicio) / 100 * 1000  # ms por búsqueda
    tiempos_search.append(t_search)
    
    # Medir tiempo de autocompletado (100 búsquedas de 3 caracteres)
    prefijos = [p[:3] for p in random.sample(palabras, min(100, n)) if len(p) >= 3]
    inicio = time.perf_counter()
    for pref in prefijos:
        trie.autocomplete(pref, max_suggestions=10)
    t_auto = (time.perf_counter() - inicio) / len(prefijos) * 1000 if prefijos else 0
    tiempos_autocomplete.append(t_auto)
    
    print(f'  n={n:>6}: insert={t_insert:.4f}s | '
          f'search={t_search:.4f}ms/op | '
          f'autocomplete={t_auto:.4f}ms/op')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Tiempo de inserción total
axes[0].plot(tamaños, tiempos_insert, 'bo-', label='Tiempo total inserción', linewidth=2, markersize=8)
# Referencia O(n*m) donde m es longitud promedio ~10
m_prom = 10
scale = tiempos_insert[0] / (tamaños[0] * m_prom)
axes[0].plot(tamaños, [scale * n * m_prom for n in tamaños], 'r--',
             label='O(n×m) referencia', alpha=0.7)
axes[0].set_xlabel('n (palabras en el Trie)')
axes[0].set_ylabel('Tiempo (segundos)')
axes[0].set_title('Tiempo de Inserción vs Tamaño')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grafico 2: Tiempo por operación (search vs autocomplete)
axes[1].plot(tamaños, tiempos_search, 'go-', label='search() ms/op', linewidth=2, markersize=8)
axes[1].plot(tamaños, tiempos_autocomplete, 'rs-', label='autocomplete() ms/op', linewidth=2, markersize=8)
axes[1].axhline(y=100, color='orange', linestyle='--', alpha=0.7, label='100ms (límite UX)')
axes[1].set_xlabel('n (palabras en el Trie)')
axes[1].set_ylabel('Tiempo (ms) por operación')
axes[1].set_title('Tiempo por Operación — Search vs Autocomplete')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Análisis Trie — Autocompletado de Búsqueda Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(INFORME_DIR / 'trie_performance.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Demo interactivo con catálogo Netflix real
print('🎬 Demo: Autocompletado con Catálogo Netflix')
print('=' * 50)

trie_netflix = Trie()
catalogo = [
    ('Stranger Things', 89), ('Squid Game', 111), ('La Casa de Papel', 75),
    ('The Crown', 60), ('Dark', 40), ('Narcos', 45), ('Black Mirror', 32),
    ('Bridgerton', 55), ('Lupin', 35), ('Peaky Blinders', 48),
    ('Breaking Bad', 70), ('Better Call Saul', 52), ('Ozark', 25),
    ('You', 38), ('Elite', 38), ('Money Heist', 75), ('The Witcher', 50),
    ('Cobra Kai', 35), ('Daredevil', 44), ('Mindhunter', 28),
    ('Bird Box', 45), ('Extraction', 99), ('Don\'t Look Up', 60)
]

for titulo, pop in catalogo:
    trie_netflix.insert(titulo, popularity=pop)

print(f'\n📚 Catálogo cargado: {trie_netflix.size()} títulos')
print()

# Simular búsquedas del usuario
busquedas = ['b', 'bl', 'bla', 'black', 'black m', 'the', 'str', 'don']
print('Simulando búsqueda del usuario escribiendo "black mirror"...')
for query in busquedas:
    resultados = trie_netflix.autocomplete(query, max_suggestions=3)
    print(f'  ["{query}"] → {[r[0] for r in resultados]}')

## 📊 Análisis de Complejidad

| Operación | Complejidad Temporal | Complejidad Espacial |
|-----------|---------------------|---------------------|
| `insert(word, popularity)` | **O(m)** | O(m × Σ) |
| `search(word)` | **O(m)** | O(1) |
| `autocomplete(prefix, k)` | **O(m + n_sub)** | O(k) |
| `delete(word)` | **O(m)** | O(1) |
| Espacio total | — | **O(N × m × Σ)** |

donde:
- `m` = longitud de la palabra/prefijo
- `n_sub` = nodos del subárbol del prefijo
- `Σ` = tamaño del alfabeto (26 letras + espacios ≈ 30)
- `N` = número de palabras

### Observación Clave

**El tiempo de búsqueda NO depende del tamaño del Trie (N)**, solo de la longitud del prefijo (m). Esto es lo que hace al Trie tan eficiente para autocompletado: agregar un millón de títulos no hace más lenta la búsqueda por prefijo.